# Lab 1a — What is an image?

**Applied Computer Vision, TU Berlin — 2026**

In this notebook you will treat a photograph not as a file but as a numerical array. You will load images captured on your Pi, inspect their shape and data type, convert between colour representations, decompose the channels, and measure how much is thrown away by JPEG compression.

**Prerequisites.** You have captured three images on your Raspberry Pi with `rpicam-still` at different resolutions (default, 640×480, 1920×1080) and copied them to your laptop via `scp`. On some older images, `libcamera-still` may also work. See the Lab 1a PDF for the capture and transfer steps.

**How this notebook is structured.** Each section has a short teaching cell followed by one or more exercise cells. Exercise cells contain scaffolded code with `???` placeholders — replace each placeholder using the documentation linked in the teaching cell. For every cell: predict what will happen before you run it, run it, then reconcile prediction with observation.

**Deliverables.** See §10. Submit the completed notebook with all placeholders replaced and all cells executed.


In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

print("OpenCV :", cv2.__version__)
print("NumPy  :", np.__version__)


In [ ]:
# Edit these three paths to point at the images you transferred from the Pi.
PATH_DEFAULT = "/Users/you/Downloads/default.jpg"
PATH_SMALL   = "/Users/you/Downloads/small.jpg"
PATH_LARGE   = "/Users/you/Downloads/large.jpg"

for p in (PATH_DEFAULT, PATH_SMALL, PATH_LARGE):
    assert os.path.exists(p), f"File not found: {p}"
print("All three files are reachable.")


## 1. How is an image stored?

At the level of a computer, an image is a multidimensional array of numbers. For a greyscale image, each pixel is a single number giving a brightness value. For a colour image, each pixel is a triple — one number per colour channel. The array is indexed by `(row, column)`, with the origin `(0, 0)` at the **top-left** pixel; for colour, a third axis indexes the channel.

Three properties of this array are load-bearing throughout the rest of the course:

- **Shape** — the `(height, width, channels)` tuple. Width is the number of columns; height is the number of rows.
- **Data type** — how each pixel value is stored in memory. `uint8` gives 256 discrete levels per channel, one byte per value. Other dtypes (`uint16`, `float32`) trade memory for range or precision.
- **Channel order** — the semantic meaning of each index along the channel axis. OpenCV reads colour images into `(B, G, R)` order. Most other libraries use `(R, G, B)`. Knowing which convention a given array follows is necessary to display and manipulate it correctly.


## 2. Loading and inspection

### 2.1 Load the three images

Use [`cv2.imread`](https://docs.opencv.org/4.x/d4/da8/group__imgcodecs.html#ga288b8b3da0892bd651fce07b3bbd3a56) to load each file into a NumPy array. The default flag returns a three-channel colour image.


In [ ]:
img_default = ???
img_small   = ???
img_large   = ???

for name, img in [("default", img_default), ("small", img_small), ("large", img_large)]:
    assert img is not None, f"{name} failed to load — check the path"
print("Loaded three images.")


### 2.2 Report shape, dtype, range, and memory

NumPy arrays expose their structure through attributes. From the [ndarray attributes documentation](https://numpy.org/doc/stable/reference/arrays.ndarray.html#array-attributes), identify the attributes that give: the shape tuple, the data type, and the total byte count in memory. NumPy also provides methods to compute the minimum and maximum values of an array.

Replace the `???` placeholders below. The output format is specified as a comment above the loop so you can check your values.


In [ ]:
# Expected output format (your numbers will differ):
#   default  shape=(H, W, 3)  dtype=uint8  min=0  max=255  memory=X.XX MB
#   small    shape=(H, W, 3)  dtype=uint8  min=...
#   large    shape=(H, W, 3)  dtype=uint8  min=...

for name, img in [("default", img_default), ("small", img_small), ("large", img_large)]:
    shape  = img.???
    dtype  = img.???
    mn, mx = img.???(), img.???()
    nbytes = img.???
    print(f"{name:8s} shape={shape} dtype={dtype} min={mn:3d} max={mx:3d} memory={nbytes/1e6:.2f} MB")


**Answer inline below (one sentence each):**

1. Which number in `shape` is the image height, and which is the width?
2. Why is height listed *before* width, rather than the `(x, y)` order used in mathematics?
3. Given the shape and dtype of the `default` image, does the `memory` value follow from a simple multiplication? Write the arithmetic out explicitly.
4. `uint8` values range over 0–255. How many distinct shades of red can be represented at a single pixel?

*Your answers:*


## 3. Displaying the image

### 3.1 Display the loaded array

Use [`matplotlib.pyplot.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.imshow.html) to render the default image. Do not convert anything yet — display it exactly as `cv2.imread` returned it.


In [ ]:
plt.figure(figsize=(8, 5))
plt.imshow(img_default)
plt.title("img_default as loaded by cv2.imread")
plt.axis('off')
plt.show()


### 3.2 Reconcile what you just saw

The colours are wrong: blue regions of the scene render red, red regions render blue, and the overall cast is shifted.

OpenCV stores pixels in **BGR** channel order — the first index along the channel axis is blue, the last is red. `matplotlib.pyplot.imshow` interprets a three-channel array as **RGB** — it assumes the first index is red. Neither library is wrong; they follow different conventions, and the array bytes are identical in both interpretations. Only the rendering differs.

**A note on colour spaces.** "RGB" and "BGR" are two orderings of the same underlying sRGB colour space — swapping between them is an index reshuffle and discards no information. A genuinely different colour space such as HSV or HSL re-parameterises the same colour on a different basis: hue (the dominant wavelength), saturation (how pure the colour is), and value or lightness. RGB is convenient for storage and display; HSV and HSL are convenient when the operation you want to perform is easier in perceptual terms ("reddish", "pale") — for example, thresholding by hue. `cv2.cvtColor` converts between any of them. See Figure 1 of the Lab 1a PDF for a side-by-side illustration of the three spaces.

Use [`cv2.cvtColor`](https://docs.opencv.org/4.x/d8/d01/group__imgproc__color__conversions.html) with the appropriate [colour-conversion code](https://docs.opencv.org/4.x/d8/d01/group__imgproc__color__conversions.html#ga4e0972be5de079fed4e3a10e24ef5ef0) to produce an RGB-ordered array, then display both side by side.


In [ ]:
img_rgb = cv2.cvtColor(img_default, cv2.???)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(img_default); axes[0].set_title("BGR array, shown as RGB"); axes[0].axis('off')
axes[1].imshow(img_rgb);     axes[1].set_title("RGB array, shown as RGB"); axes[1].axis('off')
plt.show()


## 4. Decomposing the channels

### 4.1 Separate the three colour planes

A colour image is three greyscale images stacked. Split `img_rgb` into its red, green, and blue planes and display each as a greyscale image.

Two equivalent approaches:

- NumPy slicing: `img[:, :, k]` for channel index `k`.
- [`cv2.split`](https://docs.opencv.org/4.x/d2/de8/group__core__array.html#ga0547c7fed86152d7e9d0096029c8518a).

Pick one. When you display a 2D array with `plt.imshow`, matplotlib applies its default "viridis" colormap (purple-to-yellow) — this has nothing to do with what the channel represents. To render a single channel as actual greyscale, pass `cmap='gray'`.


In [ ]:
R = img_rgb[:, :, ???]
G = img_rgb[:, :, ???]
B = img_rgb[:, :, ???]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, plane, name in zip(axes, [R, G, B], ["Red", "Green", "Blue"]):
    ax.imshow(plane, cmap='gray', vmin=0, vmax=255)
    ax.set_title(f"{name} channel  shape={plane.shape}")
    ax.axis('off')
plt.show()


**Question.** Pick a region in your scene that is predominantly one colour (a sweater, a book cover, a piece of fruit). Predict which channel will be brightest in that region, then verify by inspecting the three plots. Does it match your prediction?

*Your answer:*


### 4.2 Each pixel is three numbers — a visual grounding

Eight random pixels from the image, rendered as colour swatches alongside their `(R, G, B)` values.


In [ ]:
rng = np.random.default_rng(seed=0)
n = 8
rows = rng.integers(0, img_rgb.shape[0], n)
cols = rng.integers(0, img_rgb.shape[1], n)

fig, axes = plt.subplots(1, n, figsize=(14, 2))
for ax, r, c in zip(axes, rows, cols):
    rgb = img_rgb[r, c]
    ax.imshow(np.full((10, 10, 3), rgb, dtype=np.uint8))
    ax.set_title(f"({rgb[0]},{rgb[1]},{rgb[2]})", fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout()
plt.show()


Each coloured square above is a single pixel from your image, drawn large enough to see. The three numbers underneath are its `(R, G, B)` values — exactly the triple stored in memory at that `(row, column)`. The colour you perceive is a direct consequence of those three numbers.

### 4.3 Your image as a cloud in RGB space

An image can be seen as a collection of points in a three-dimensional colour space, with one point per pixel. Below, a random subsample of pixels is scattered inside the RGB cube, each point coloured as itself. The shape of the cloud tells you which colours dominate the scene.


In [ ]:
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

N = 3000  # subsample for speed and legibility
flat = img_rgb.reshape(-1, 3)
idx  = np.random.default_rng(seed=1).choice(flat.shape[0], size=N, replace=False)
pixels = flat[idx]

fig = plt.figure(figsize=(7, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(pixels[:, 0], pixels[:, 1], pixels[:, 2],
           c=pixels / 255.0, s=4)
ax.set_xlabel('R'); ax.set_ylabel('G'); ax.set_zlabel('B')
ax.set_xlim(0, 255); ax.set_ylim(0, 255); ax.set_zlim(0, 255)
ax.set_title(f"RGB distribution of {N} pixels sampled from img_rgb")
plt.show()


**Question.** Does the cloud hug one axis (most pixels concentrated along green, say), spread along the main diagonal (mostly near-grey pixels — a scene with little colour saturation), or form distinct clusters (a scene with a few dominant colour regions)? Write one sentence relating the shape of the cloud to what you can see in the photograph.

*Your answer:*


## 5. Converting to greyscale

### 5.1 Two ways to collapse colour to intensity

A greyscale image has one intensity value per pixel instead of three. Two natural ways to produce it from a colour image:

- **Perceptual weighting.** The human eye is more sensitive to green than to red or blue. A weighted sum $0.299\,R + 0.587\,G + 0.114\,B$ (the ITU-R BT.601 luma coefficients) approximates perceived brightness. This is what [`cv2.cvtColor`](https://docs.opencv.org/4.x/d8/d01/group__imgproc__color__conversions.html) produces when converting from RGB or BGR to `GRAY`.
- **Simple mean.** $(R + G + B) / 3$ — all channels equally weighted. Simpler, perceptually inaccurate.

Compute both, then compare.


In [ ]:
gray_perceptual = cv2.cvtColor(img_rgb, cv2.???)
gray_mean       = img_rgb.???(axis=???).astype(np.???)

print("gray_perceptual shape:", gray_perceptual.shape, " dtype:", gray_perceptual.dtype)
print("gray_mean       shape:", gray_mean.shape,       " dtype:", gray_mean.dtype)


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img_rgb);                        axes[0].set_title("Original (RGB)");         axes[0].axis('off')
axes[1].imshow(gray_perceptual, cmap='gray');   axes[1].set_title("Perceptual greyscale");   axes[1].axis('off')
axes[2].imshow(gray_mean, cmap='gray');         axes[2].set_title("Mean greyscale");         axes[2].axis('off')
plt.show()


**Observations.**

- The shape changed from `(H, W, 3)` to `(H, W)`. The channel axis is gone because greyscale uses a single number per pixel.
- The two greyscale images differ, usually most visibly in regions dominated by red or blue. The perceptual weighting pushes red down and green up relative to the naïve mean, better matching how a human ranks the brightness of those regions.


## 6. Data types

### 6.1 `uint8`, `uint16`, `float32`

Image arrays can be stored in different numerical types. The dtype fixes both how much memory each pixel occupies and what operations on it are valid.

- **`uint8`** — 1 byte per value, range 0–255. Standard output of consumer cameras. 256 brightness levels per channel.
- **`uint16`** — 2 bytes per value, range 0–65535. Output of scientific and industrial cameras (often 10- or 12-bit sensors padded to 16 bits). Double the memory, but preserves shadow detail that `uint8` quantises away.
- **`float32`** — 4 bytes per value. What we cast to whenever we *compute* with an image: filtering, averaging, gradient estimation, neural network input. Two reasons we cast before computing: `uint8` overflows silently when a sum exceeds 255 (so `200 + 100` wraps to `44`, not `300`), and `uint8` cannot represent fractional values at all. Casting to `float32` gives both the headroom and the sub-integer precision needed for intermediate computation. By convention, `float32` image data is normalised to `[0, 1]`, so multiplying by `0.5` darkens the image rather than moving values out of range.

Convert `gray_perceptual` to each of the three dtypes and report their dtype, range, and memory.


In [ ]:
g8  = gray_perceptual.astype(np.???)
g16 = gray_perceptual.astype(np.???) * 257         # rescale 0..255 -> 0..65535
gf  = gray_perceptual.astype(np.???) / 255.0       # rescale 0..255 -> 0..1

for name, arr in [("uint8", g8), ("uint16", g16), ("float32", gf)]:
    print(f"{name:8s}  dtype={arr.dtype}  min={arr.min():10.4f}  max={arr.max():10.4f}  memory={arr.nbytes/1e6:.2f} MB")


**A quick overflow demonstration.** Run the cell below to see the silent wrap-around first-hand. This is why we cast before any arithmetic.


In [ ]:
a = np.array([200], dtype=np.uint8)
b = np.array([100], dtype=np.uint8)
print("uint8  200 + 100 =", (a + b)[0], "  (wrapped, because 300 > 255)")
print("float32 200 + 100 =", (a.astype(np.float32) + b.astype(np.float32))[0], "  (correct)")


## 7. Cropping and indexing

### 7.1 An image is a NumPy array — slice it

Because the image is an array, you can crop with standard slicing: `img[row_start:row_stop, col_start:col_stop]`. The first index selects rows — the vertical position, measured downward from the top. The second selects columns — the horizontal position, measured rightward from the left. Index `(0, 0)` is the top-left pixel. This row-first indexing is the reverse of the `(x, y)` convention from mathematics, and is a recurring source of off-by-one errors until you internalise it.

Take two crops of different sizes from an interesting region of your image. Observe how the shape of the array changes with the region you keep.


In [ ]:
print(f"Original: shape={img_rgb.shape}")

crop_large = img_rgb[200:800, 400:1400]     # adjust ranges for your image
crop_small = img_rgb[400:500, 700:900]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img_rgb);    axes[0].set_title(f"full        {img_rgb.shape[:2]}");    axes[0].axis('off')
axes[1].imshow(crop_large); axes[1].set_title(f"large crop  {crop_large.shape[:2]}"); axes[1].axis('off')
axes[2].imshow(crop_small); axes[2].set_title(f"small crop  {crop_small.shape[:2]}"); axes[2].axis('off')
plt.show()

print(f"Large crop: shape={crop_large.shape}")
print(f"Small crop: shape={crop_small.shape}")


Each crop's shape matches exactly the number of rows and columns selected. Cropping changes the array size but not the pixel density: fewer pixels at the same spatial resolution, covering a smaller portion of the scene. Downsampling (which *does* change pixel density — one output pixel aggregates several input pixels) is a different operation; see [`cv2.resize`](https://docs.opencv.org/4.x/da/d54/group__imgproc__transform.html#ga47a974309e9102f5f08231edc7e7529d).


## 8. Raw array size vs on-disk file size

Your three images have very different on-disk sizes and very different array sizes. Compute both for each.

Array size in memory: `height × width × channels × bytes_per_value`. For a `uint8` colour image, that is `H × W × 3` bytes.

File size on disk: use [`os.path.getsize`](https://docs.python.org/3/library/os.path.html#os.path.getsize).

The ratio between the two is the **compression ratio** — how aggressively the JPEG encoder shrank the array to produce the file you stored.


In [ ]:
for name, img, path in [
    ("default", img_default, PATH_DEFAULT),
    ("small",   img_small,   PATH_SMALL),
    ("large",   img_large,   PATH_LARGE),
]:
    H, W, C  = img.shape
    raw      = H * W * C                 # bytes, since uint8 = 1 byte per value
    on_disk  = os.path.getsize(path)
    ratio    = raw / on_disk
    print(f"{name:8s}  {W}x{H}  raw={raw/1e6:6.2f} MB  on_disk={on_disk/1e6:6.2f} MB  compression = {ratio:5.1f}x")


**What does JPEG discard to achieve this compression?** JPEG applies a two-dimensional discrete cosine transform to 8×8 blocks of the image, quantises the high-frequency coefficients aggressively, and entropy-codes the result. The high-frequency components it discards correspond to fine texture and sharp edges, which the human visual system is less sensitive to than smooth gradients. This is lossy compression: the decoded image is not bit-identical to the input, only perceptually close.

This point returns in Lab 1b, where compression ratio directly determines whether the video bitstream fits the available bandwidth.


## 9. Saving at different quality levels

Use [`cv2.imwrite`](https://docs.opencv.org/4.x/d4/da8/group__imgcodecs.html#ga8ac397bd09e48851665edbe12aa28f25) to save the same image four times: once as PNG (lossless) and three times as JPEG at qualities 90, 50, and 10. Compare on-disk sizes, then look at the quality-10 JPEG.

`cv2.imwrite` writes from the BGR convention, so pass the original `img_default` (not `img_rgb`) to preserve colours in the saved file.


In [ ]:
out_dir = "./out"
os.makedirs(out_dir, exist_ok=True)

cv2.imwrite(f"{out_dir}/out.png",     img_default)
cv2.imwrite(f"{out_dir}/out_q90.jpg", img_default, [cv2.IMWRITE_JPEG_QUALITY, ???])
cv2.imwrite(f"{out_dir}/out_q50.jpg", img_default, [cv2.IMWRITE_JPEG_QUALITY, ???])
cv2.imwrite(f"{out_dir}/out_q10.jpg", img_default, [cv2.IMWRITE_JPEG_QUALITY, ???])

for fname in ["out.png", "out_q90.jpg", "out_q50.jpg", "out_q10.jpg"]:
    size = os.path.getsize(f"{out_dir}/{fname}")
    print(f"{fname:12s}  {size/1e6:6.3f} MB")


In [ ]:
q10 = cv2.cvtColor(cv2.imread(f"{out_dir}/out_q10.jpg"), cv2.COLOR_BGR2RGB)
q90 = cv2.cvtColor(cv2.imread(f"{out_dir}/out_q90.jpg"), cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(q90); axes[0].set_title("quality = 90"); axes[0].axis('off')
axes[1].imshow(q10); axes[1].set_title("quality = 10"); axes[1].axis('off')
plt.show()


**Observations.** The quality-10 JPEG shows blocky artefacts at edges and loses fine texture. The PNG is larger than the quality-90 JPEG because PNG is lossless — it compresses but discards nothing. The quality parameter is a direct lever on the rate–distortion tradeoff: lower quality yields a smaller file with more visible artefacts.


## 10. Takeaways and homework

### 10.1 Write your own takeaways

Writing down what you learned in your own words is not busywork. Three things happen when you do:

1. **Active recall.** Reconstructing the material from memory cements it far more effectively than re-reading does. The act of producing a sentence is the learning.
2. **Gap detection.** You discover which pieces you cannot yet articulate, which tells you precisely what to revisit before the next lab.
3. **A searchable record.** When BGR vs RGB, `float32` normalisation, or compression returns in Lab 3 or Lab 7, your own notes are faster and more useful to skim than the original PDF.

Fill in the template below. Two to three sentences per prompt is plenty. Be specific and honest — there is no reward for pretending to understand something you don't.

---

**Before this session, I thought an image was:**

*(your answer)*

**Now I think an image is:**

*(your answer)*

**One thing that surprised me today:**

*(your answer)*

**One thing I want to revisit or don't fully understand yet:**

*(your answer)*

**A question this session raised that we didn't answer:**

*(your answer)*

**Other takeaways worth writing down for future-me:**

*(your answer — anything from a neat piece of code, to a concept that clicked, to a wrong prediction you made along the way)*


### 10.2 Homework — grayscale histogram on a home image

When you take your Pi home, configure it for your home Wi-Fi (see Appendix B of the Lab 1a PDF) and capture two or three additional images of scenes you find interesting. Transfer one to your laptop.

Below, load the home image, convert it to greyscale using the perceptual method, and plot its histogram over the 256 possible intensity values.

Reference: [`matplotlib.pyplot.hist`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.hist.html), [`numpy.ndarray.ravel`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.ravel.html).

In 2–3 sentences below the plot, relate the histogram shape to the scene's exposure. A histogram skewed low suggests an underexposed (dark) image; skewed high, overexposed (bright); concentrated in the middle, well-exposed with little dynamic range; spread broadly, well-exposed with high dynamic range.


In [ ]:
PATH_HOME = "/path/to/your/home_image.jpg"
img_home  = cv2.imread(PATH_HOME)
gray_home = cv2.cvtColor(img_home, cv2.???)

plt.figure(figsize=(8, 3))
plt.hist(gray_home.???(), bins=256, range=(0, 256))
plt.xlabel("intensity (0–255)"); plt.ylabel("pixel count")
plt.title("Grayscale histogram — home image")
plt.show()


*Your exposure analysis (2–3 sentences):*


### Submission

Submit this notebook with every `???` placeholder replaced, all cells executed, and the takeaways template in §10.1 filled in. Rerun from a fresh kernel before submission so the cell outputs reflect the final state of your code.
